In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from glob import glob

In [ ]:
active_space_df = pd.read_csv("../DDLUCJ_active_spaces_unfrozen.csv",delimiter=';')

In [ ]:
# Clean the '.xyz' suffix and convert to dictionary
mol_dict = (
    active_space_df.assign(xyz=active_space_df["xyz"].str.replace(".xyz", ""))
    .set_index("xyz")["molecule"]
    .to_dict()
)

In [ ]:
dmrg_data = []
for _, row in active_space_df.iterrows():
    rowdict = row.to_dict()
    molecule = rowdict['molecule']
    print(molecule)
    for i in glob(f"*/{molecule}/dmrg_Energy.csv"):
        basis, name, filename = i.split("/")
        datadf = pd.read_csv(i)
        dmrg_data.append(('DMRG-CASCI',basis, name,np.squeeze(datadf['energy'])))
dmrg_data = pd.DataFrame(dmrg_data,columns=['Method','Basis Set','Name','Energy'])

In [ ]:
df = pd.read_csv('energies.csv', index_col=0)

# # Replace GDB names (e.g. GDB04_33) with full molecule names using your dictionary
# df['Name'] = df['Name'].map(mol_dict).fillna(df['Name'])

# df = pd.concat([df,dmrg_data])
# df.drop_duplicates().to_csv('energies.csv')

In [ ]:
diffdf = []
for basis in df['Basis Set'].unique():
    for molecule in df['Name'].unique():
        subset = df[(df['Basis Set'] == basis) & (df['Name'] == molecule)]
        subset.loc[:, 'Energy'] = subset.loc[:, 'Energy'].values - subset.loc[subset['Method'] == 'DMRG-CASCI', 'Energy'].values
        diffdf.append(subset)

diffdf = pd.concat(diffdf)        

In [ ]:
df

In [ ]:
sns.catplot(df,x='Name',y='Energy',hue='Method',col='Basis Set',kind='bar')